# What are Cross-validation , K-fold Cross-Validation ,Cross_val_score , GridSearchCV

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold , cross_val_score , GridSearchCV
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [10]:
X = df.drop('Outcome',axis= 1)
y = df['Outcome']
print(X.shape , y.shape)

# K-Fold Cross-Validation
print("================================K-Fold EXAMPLE================")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
for train_index, test_index in kf.split(X):
    print(f'Fold {fold} :')
    print('Train indices :',train_index[:5])
    print('Test indices: ',test_index[:5])
    fold += 1

# KFold itself doesnot train anything - just gives train/test indices
# =====================================
# Cross_val_score
# =======================================
print('================================CROSS VAL SCORE EXAMPLE  =========================')
lg = LogisticRegression(max_iter=1000)
scores = cross_val_score(lg , X, y ,cv=kf , scoring ='accuracy')
print('Fold - wise accuracy :',scores)
print('mean accuracy :',scores.mean())

(768, 8) (768,)
================================K-Fold EXAMPLE================
Fold 1 :
Train indices : [0 1 3 4 5]
Test indices:  [ 2  7 10 23 30]
Fold 2 :
Train indices : [0 1 2 3 4]
Test indices:  [ 6 11 15 18 24]
Fold 3 :
Train indices : [1 2 3 4 5]
Test indices:  [ 0  9 12 17 19]
Fold 4 :
Train indices : [0 1 2 4 6]
Test indices:  [ 3  5  8 16 26]
Fold 5 :
Train indices : [0 2 3 5 6]
Test indices:  [ 1  4 13 14 20]
================================CROSS VAL SCORE EXAMPLE  =========================
Fold - wise accuracy : [0.74675325 0.78571429 0.74675325 0.81045752 0.75163399]
mean accuracy : 0.7682624564977505


In [11]:
param_grid = {
    'penalty' : ['l1' ,'l2' , 'elasticnet' ,None],
    'C' :[0.001 , 0.01 , 0.1 , 1 ,10 , 100],
    'solver' : ['lbfgs' ,'newton-cg' ,'liblinear' ,'sag','saga'],
    'max_iter' : [100,200,500, 1000, 2500 , 5000]
    
}

grid_search = GridSearchCV(
    estimator = lg,
    param_grid = param_grid,
    cv = 5 ,
    scoring = 'accuracy',
    verbose = 1 ,
    n_jobs = -1
)
grid_search.fit(X, y )
print('\n Best Parameters Found :',grid_search.best_params_)
print('\Best Cross-val score (MSE) :',grid_search.best_score_)
print("\n======== SUMMARY ========")
print("""
K-Fold:        → Only defines how to split data (no training).
cross_val_score: → Evaluates a model using cross-validation (gives scores for fixed params).
GridSearchCV:     → Tunes hyperparameters using cross-validation (finds best params).
""")

<>:19: SyntaxWarning: invalid escape sequence '\B'
<>:19: SyntaxWarning: invalid escape sequence '\B'
C:\Users\himan\AppData\Local\Temp\ipykernel_3056\1640705765.py:19: SyntaxWarning: invalid escape sequence '\B'
  print('\Best Cross-val score (MSE) :',grid_search.best_score_)


Fitting 5 folds for each of 720 candidates, totalling 3600 fits

 Best Parameters Found : {'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'newton-cg'}
\Best Cross-val score (MSE) : 0.7721925133689839

======== SUMMARY ========

K-Fold:        → Only defines how to split data (no training).
cross_val_score: → Evaluates a model using cross-validation (gives scores for fixed params).
GridSearchCV:     → Tunes hyperparameters using cross-validation (finds best params).



C:\Users\himan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
1620 fits failed out of a total of 3600.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\himan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\himan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\himan\A

In [ ]:
# Cross - Validation Score
# A method to evalute how well a model generalizes :-
# Insetead of splitting data once (train/test), we split it k-times- each time training 
# on (k-1) parts and testing on the remaining one.
# Each fold gives a validataion score and we take the average -> that's our cross-validation score
# fold                  train on          teston
# 1                       Folds 2-5          Fold 1
# 2                       Folds 1 ,3-5       Fold 2
# 3                       Folds 1-2 ,4-5     Fold 3
# 4                       Fold 1-3,5         Fold 4
# 5                       fold 1-4           Fold 5

# K-Fold Cross Validation 
# k - Fold is the manual technique that divides your dataset into k equal folds , trains on (k-1)
# , and validates on the remaining one .
# It's the  mechanism used by both cross_val_score()  and GridSearch CV.


# K-fold cross-validation is a machine learning technique that divides a dataset into 
# 'k' equal-sized folds. The model is trained on 'k-1' folds and validated on the remaining
# fold, repeating this process 'k' times so each fold is used for validation once. The 
# final performance metric is the average of the results from all 'k' iterations, which 
# provides a more robust estimate of the model's generalization ability. 

# Shuffle the data: The dataset is first randomly shuffled.
# Split into folds: The shuffled dataset is divided into 'k' equal-sized folds. 
# For example, with 5-fold cross-validation, the data would be split into 5 groups of 
# equal size.
# Iterate and validate: The process is repeated 'k' times:
# One fold is held out as the test set.
# The remaining 'k-1' folds are combined to form the training set.
# The model is trained on the training set and then evaluated on the test set.
# The result (like accuracy or error) is recorded.
# Average the results: After all 'k' iterations are complete, the performance metric 
# from each iteration is averaged to get the final cross-validation score. 
# Why use k-fold cross-validation?
# Prevents overfitting: By testing on different subsets of the data, it gives a more 
# reliable estimate of how the model will perform on unseen data compared to a single 
# train-test split.
# Uses all data: Every data point is used for both training and validation at some 
# point in the process, so the final evaluation is based on the entire dataset.
# Robust performance estimate: Averaging the results across multiple folds provides a 
#     more stable and trustworthy measure of model performance. 